# Nombre del estudiante:

Daniel Guanga

# Computacion: 

Computacion

# Materia:

Inteligencia Artificial

# K-Nearest Neighbors (KNN) con Transformación de Características

Este notebook implementa el algoritmo K-Nearest Neighbors desde cero para predecir diabetes en pacientes ecuatorianos.

## Contenido:
1. Instalación de dependencias
2. Importar librerías necesarias
3. Definir la clase FeatureTransformer
4. Definir funciones de distancia euclidiana
5. Implementar el algoritmo KNN
6. Cargar y explorar los datos
7. Preparar la etiqueta target
8. Transformar las características
9. Calcular K-Nearest Neighbors
10. Análisis de resultados y predicción

## 1. Instalación de Dependencias

Instalaremos automáticamente las librerías necesarias (pandas, numpy, scikit-learn) si no están disponibles.

In [1]:
# Instalar automáticamente las librerías necesarias si no están disponibles
import sys
import subprocess

def ensure_packages():
    """Instala automáticamente las librerías necesarias si no están disponibles."""
    try:
        import pandas as _p
        import numpy as _n
    except Exception:
        # Si pandas o numpy no están instalados, los instala
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'numpy'])
    try:
        from sklearn.preprocessing import StandardScaler as _scl
    except Exception:
        # Si scikit-learn no está instalado, lo instala
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn'])

ensure_packages()
print("✓ Dependencias instaladas correctamente")

✓ Dependencias instaladas correctamente


## 2. Importar Librerías Necesarias

Importamos pandas para manipulación de datos, numpy para cálculos numéricos y typing para anotaciones de tipos.

In [2]:
import pandas as pd
import numpy as np
from typing import List, Tuple
from pathlib import Path

print("✓ Librerías importadas:")
print(f"  - pandas {pd.__version__}")
print(f"  - numpy {np.__version__}")

✓ Librerías importadas:
  - pandas 3.0.2
  - numpy 2.4.4


## 3. Definir la Clase FeatureTransformer

Esta clase transforma datos categóricos y numéricos en un formato numérico compatible con KNN.

**Pasos:**
1. One-hot encoding para variables categóricas (ciudad, sexo)
2. Encoding ordinal para colesterol (bajo → 0, medio → 1, alto → 2, muy alto → 3)
3. Estandarización de la edad (media 0, desviación estándar 1)

In [3]:
class FeatureTransformer:
    """
    Clase para transformar datos categóricos y numéricos en un formato numérico
    compatible con el algoritmo KNN.
    """
    
    def __init__(self):
        """Inicializa los parámetros que se aprenderán del dataset de entrenamiento."""
        # Guardará los nombres de las columnas dummy generadas para ciudad
        self.city_cols = None
        # Guardará los nombres de las columnas dummy generadas para sexo
        self.sex_cols = None
        # Guardará la media de la edad del conjunto de entrenamiento
        self.edad_mean = None
        # Guardará la desviación estándar de la edad del conjunto de entrenamiento
        self.edad_std = None
        # Diccionario para convertir colesterol de texto a número ordinal
        # bajo=0, medio=1, alto=2, muy alto=3
        self.chol_map = {'bajo': 0, 'medio': 1, 'alto': 2, 'muy alto': 3}

    def fit(self, df: pd.DataFrame):
        """
        Aprende la estructura de los datos: 
        - Qué ciudades existen
        - Qué valores de sexo existen
        - Media y desviación estándar de la edad
        """
        # Crea variables dummy (one-hot encoding) para las ciudades
        city_dummies = pd.get_dummies(df['ciudad'], prefix='ciudad')
        # Crea variables dummy para sexo (convierte a string primero)
        sex_dummies = pd.get_dummies(df['sexo'].astype(str), prefix='sexo')
        
        # Guarda los nombres de las columnas para usarlas luego en transform()
        self.city_cols = list(city_dummies.columns)
        self.sex_cols = list(sex_dummies.columns)
        
        # Calcula la media de la edad
        self.edad_mean = df['edad'].mean()
        # Calcula la desviación estándar poblacional (ddof=0)
        # Esto coincide con StandardScaler de sklearn
        self.edad_std = df['edad'].std(ddof=0) if df['edad'].std(ddof=0) != 0 else 1.0

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Transforma los datos crudos a un formato numérico:
        1. Mapea colesterol a valores ordinales
        2. Crea variables dummy para ciudad y sexo
        3. Estandariza la edad usando los parámetros aprendidos en fit()
        4. Retorna una matriz numérica lista para KNN
        """
        # Crea una copia para no modificar el dataframe original
        df = df.copy()
        
        # Convierte colesterol de texto a número usando el diccionario chol_map
        df['colesterol_ord'] = df['colesterol'].map(self.chol_map).astype(float)
        
        # Crea variables dummy para ciudad en los nuevos datos
        city_dummies = pd.get_dummies(df['ciudad'], prefix='ciudad')
        # Crea variables dummy para sexo en los nuevos datos
        sex_dummies = pd.get_dummies(df['sexo'].astype(str), prefix='sexo')
        
        # Asegura que todas las columnas del entrenamiento estén presentes
        # Si falta una columna, la agrega con valor 0
        for c in (self.city_cols or []):
            if c not in city_dummies:
                city_dummies[c] = 0
        for c in (self.sex_cols or []):
            if c not in sex_dummies:
                sex_dummies[c] = 0
        
        # Ordena las columnas alfabéticamente para garantizar consistencia
        city_dummies = city_dummies.reindex(sorted(self.city_cols), axis=1)
        sex_dummies = sex_dummies.reindex(sorted(self.sex_cols), axis=1)

        # Estandariza la edad: (edad - media) / desviación_estándar
        # Resultado: edad con media 0 y desviación estándar 1
        df['edad_scaled'] = (df['edad'] - self.edad_mean) / self.edad_std

        # Concatena todas las características numéricas
        features = pd.concat([
            df[['edad_scaled', 'colesterol_ord']].reset_index(drop=True),
            sex_dummies.reset_index(drop=True),
            city_dummies.reset_index(drop=True)
        ], axis=1)
        return features

print("✓ Clase FeatureTransformer definida correctamente")

✓ Clase FeatureTransformer definida correctamente


## 4. Definir Funciones de Distancia Euclidiana

Implementamos la función euclidean_distances() que calcula la distancia euclídea entre un nuevo punto y todos los puntos del conjunto de entrenamiento.

**Fórmula:** $d = \sqrt{(x_1-a_1)^2 + (x_2-a_2)^2 + ... + (x_n-a_n)^2}$

In [4]:
def euclidean_distances(X: np.ndarray, x_new: np.ndarray) -> np.ndarray:
    """
    Calcula la distancia euclídea entre un nuevo punto y todos los puntos 
    del conjunto de entrenamiento.
    
    Fórmula: d = sqrt((x1-a1)^2 + (x2-a2)^2 + ... + (xn-an)^2)
    """
    # Convierte a arrays numpy de tipo float para evitar errores de tipo
    X_arr = np.asarray(X, dtype=float)
    x_arr = np.asarray(x_new, dtype=float)
    
    # Resta el nuevo punto de cada fila (broadcasting automático)
    diff = X_arr - x_arr
    
    # Calcula la raíz cuadrada de la suma de cuadrados
    return np.sqrt(np.sum(diff ** 2, axis=1))

print("✓ Función euclidean_distances definida correctamente")

✓ Función euclidean_distances definida correctamente


## 5. Implementar el Algoritmo KNN

Desarrollamos la función knn_predict() que implementa K-Nearest Neighbors:
1. Calcula distancia a todos los puntos de entrenamiento
2. Ordena por distancia (menor primero)
3. Toma los k vecinos más cercanos
4. Vota mayoritariamente para hacer la predicción

In [5]:
def knn_predict(X_train: np.ndarray, y_train: np.ndarray, x_new: np.ndarray, k: int = 3) -> Tuple[List[int], List[float], int]:
    """
    Implementa el algoritmo K-Nearest Neighbors (KNN).
    
    Parámetros:
    - X_train: matriz de características de entrenamiento
    - y_train: etiquetas (0 o 1) de entrenamiento
    - x_new: características del nuevo punto a predecir
    - k: número de vecinos a considerar (default=3)
    
    Retorna: (índices_vecinos, distancias_vecinos, predicción)
    """
    
    # Calcula las distancias del nuevo punto a todos los puntos de entrenamiento
    dists = euclidean_distances(X_train, x_new)
    
    # Obtiene los índices que ordenarían las distancias (del menor al mayor)
    idx_sorted = np.argsort(dists)
    
    # Toma los índices de los k primeros (más cercanos)
    neighbors_idx = idx_sorted[:k].tolist()
    
    # Obtiene las distancias correspondientes a esos vecinos
    neighbors_dists = dists[neighbors_idx].tolist()
    
    # Convierte y_train a array numpy
    y_arr = np.asarray(y_train)
    
    # Obtiene las etiquetas de los k vecinos más cercanos
    votes = y_arr[neighbors_idx]
    
    # Votación mayoritaria: promedio de las etiquetas redondeado
    # Si promedio <= 0.5 -> 0, si > 0.5 -> 1
    pred = int(np.round(np.mean(votes)))
    
    return neighbors_idx, neighbors_dists, pred

print("✓ Función knn_predict definida correctamente")

✓ Función knn_predict definida correctamente


## 6. Cargar y Explorar los Datos

Cargamos el archivo CSV con los datos de los pacientes ecuatorianos.

In [6]:
# Cargar el dataset
# Ajusta la ruta según tu estructura de carpetas
csv_path = Path('data.csv')

if not csv_path.exists():
    # Si el archivo no está en la carpeta actual, busca en la carpeta del notebook
    csv_path = Path('.') / 'data.csv'

df = pd.read_csv(csv_path)

print("✓ Datos cargados correctamente")
print(f"\nForma del dataset: {df.shape}")
print(f"\nDatos originales:")
print(df)

✓ Datos cargados correctamente

Forma del dataset: (6, 5)

Datos originales:
   sexo     ciudad colesterol  edad diabetes
0     1     Cuenca       bajo    18       no
1     2      Quito       alto    52       si
2     2  Guayaquil      medio    34       no
3     1       Loja       alto    61       si
4     2     Ambato      medio    45       no
5     1    Machala   muy alto    67       si


### Información del Dataset

Mostramos información detallada sobre los datos cargados.

In [7]:
print("\n" + "="*50)
print("INFORMACIÓN DEL DATASET")
print("="*50)
print(f"\nColumnas: {df.columns.tolist()}")
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nValores únicos por columna:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} valores únicos")
    if col != 'edad':
        print(f"    Valores: {df[col].unique().tolist()}")


INFORMACIÓN DEL DATASET

Columnas: ['sexo', 'ciudad', 'colesterol', 'edad', 'diabetes']

Tipos de datos:
sexo          int64
ciudad          str
colesterol      str
edad          int64
diabetes        str
dtype: object

Valores únicos por columna:
  sexo: 2 valores únicos
    Valores: [1, 2]
  ciudad: 6 valores únicos
    Valores: ['Cuenca', 'Quito', 'Guayaquil', 'Loja', 'Ambato', 'Machala']
  colesterol: 4 valores únicos
    Valores: ['bajo', 'alto', 'medio', 'muy alto']
  edad: 6 valores únicos
  diabetes: 2 valores únicos
    Valores: ['no', 'si']


## 7. Preparar la Etiqueta Target

Convertimos la variable diabetes de formato texto ('no'/'sí') a formato binario (0/1).

In [8]:
# Convierte 'no' -> 0 y 'sí' -> 1 para la variable diabetes
df['diabetes_bin'] = df['diabetes'].map({'no': 0, 'si': 1}).astype(int)

print("✓ Etiqueta target preparada")
print(f"\nEtiqueta diabetes mapeada (no=0, sí=1):")
print(df[['diabetes', 'diabetes_bin']])

✓ Etiqueta target preparada

Etiqueta diabetes mapeada (no=0, sí=1):
  diabetes  diabetes_bin
0       no             0
1       si             1
2       no             0
3       si             1
4       no             0
5       si             1


## 8. Transformar las Características

Aplicamos transformaciones a los datos:
- One-hot encoding para ciudad y sexo
- Encoding ordinal para colesterol
- Estandarización de edad

In [9]:
print("="*80)
print("TRANSFORMAR CARACTERÍSTICAS")
print("="*80)

# Crea el transformador
transformer = FeatureTransformer()
# Lo ajusta con los datos de entrenamiento (aprende estructura)
transformer.fit(df)

print("\n✓ Parámetros aprendidos:")
print(f"  - Ciudades encontradas: {transformer.city_cols}")
print(f"  - Sexos encontrados: {transformer.sex_cols}")
print(f"  - Media de edad: {transformer.edad_mean:.2f}")
print(f"  - Desv. Est. de edad: {transformer.edad_std:.2f}")

# Transforma los datos usando los parámetros aprendidos
X = transformer.transform(df)
# Extrae las etiquetas como array numpy
y = df['diabetes_bin'].to_numpy()

print("\n✓ Datos transformados correctamente")

TRANSFORMAR CARACTERÍSTICAS

✓ Parámetros aprendidos:
  - Ciudades encontradas: ['ciudad_Ambato', 'ciudad_Cuenca', 'ciudad_Guayaquil', 'ciudad_Loja', 'ciudad_Machala', 'ciudad_Quito']
  - Sexos encontrados: ['sexo_1', 'sexo_2']
  - Media de edad: 46.17
  - Desv. Est. de edad: 16.49

✓ Datos transformados correctamente


### Tabla Estandarizada

Mostramos los datos transformados en formato numérico.

In [10]:
print("\n" + "="*80)
print("TABLA ESTANDARIZADA (Datos transformados)")
print("="*80)
print(X)

print(f"\nForma de la matriz: {X.shape}")
print(f"Etiquetas (y): {y}")


TABLA ESTANDARIZADA (Datos transformados)
   edad_scaled  colesterol_ord  sexo_1  sexo_2  ciudad_Ambato  ciudad_Cuenca  \
0    -1.708466             0.0    True   False          False           True   
1     0.353824             2.0   False    True          False          False   
2    -0.737976             1.0   False    True          False          False   
3     0.899725             2.0    True   False          False          False   
4    -0.070765             1.0   False    True           True          False   
5     1.263658             3.0    True   False          False          False   

   ciudad_Guayaquil  ciudad_Loja  ciudad_Machala  ciudad_Quito  
0             False        False           False         False  
1             False        False           False          True  
2              True        False           False         False  
3             False         True           False         False  
4             False        False           False         False  
5     

## 9. Calcular K-Nearest Neighbors

Creamos un nuevo paciente y ejecutamos el algoritmo KNN para encontrar sus 3 vecinos más cercanos.

In [11]:
print("\n" + "="*80)
print("NUEVO PACIENTE A PREDECIR")
print("="*80)

# Crear un nuevo paciente con datos a predecir
nuevo = pd.DataFrame([{
    'sexo': 2,
    'ciudad': 'Cuenca',
    'colesterol': 'alto',
    'edad': 50
}])

print("\nDatos del nuevo paciente:")
print(nuevo)

# Transforma el nuevo paciente usando el mismo transformador
x_new_feat = transformer.transform(nuevo)

print("\nNuevo paciente transformado:")
print(x_new_feat)


NUEVO PACIENTE A PREDECIR

Datos del nuevo paciente:
   sexo  ciudad colesterol  edad
0     2  Cuenca       alto    50

Nuevo paciente transformado:
   edad_scaled  colesterol_ord  sexo_1  sexo_2  ciudad_Ambato  ciudad_Cuenca  \
0     0.232513             2.0       0    True              0           True   

   ciudad_Guayaquil  ciudad_Loja  ciudad_Machala  ciudad_Quito  
0                 0            0               0             0  


### Ejecutar KNN

Ejecutamos el algoritmo K-Nearest Neighbors con k=3.

In [12]:
print("\n" + "="*80)
print("CALCULAR K-NEAREST NEIGHBORS (k=3)")
print("="*80)

# Ejecuta el algoritmo KNN
neighbors_idx, neighbors_dists, pred = knn_predict(X.to_numpy(), y, x_new_feat.to_numpy()[0], k=3)

print("\n✓ KNN ejecutado correctamente")


CALCULAR K-NEAREST NEIGHBORS (k=3)

✓ KNN ejecutado correctamente


## 10. Análisis de Resultados y Predicción

Mostramos las distancias calculadas, los vecinos seleccionados y la predicción final.

In [13]:
print("\n" + "="*80)
print("RESULTADOS DEL ANÁLISIS")
print("="*80)

print('\n1. Distancias calculadas a cada vecino (k=3):')
for i, (idx, dist) in enumerate(zip(neighbors_idx, neighbors_dists), 1):
    print(f"   Vecino {i}: distancia = {dist:.4f}")

print('\n2. Vecinos seleccionados (índices):')
print(f"   {neighbors_idx}")

print('\n3. Detalles de los vecinos más cercanos:')
for i, idx in enumerate(neighbors_idx, 1):
    vecino = df.iloc[idx].to_dict()
    print(f"\n   Vecino {i} (índice {idx}, distancia {neighbors_dists[i-1]:.4f}):")
    print(f"     - Sexo: {vecino['sexo']}")
    print(f"     - Ciudad: {vecino['ciudad']}")
    print(f"     - Colesterol: {vecino['colesterol']}")
    print(f"     - Edad: {vecino['edad']}")
    print(f"     - Diabetes: {vecino['diabetes']} ({vecino['diabetes_bin']})")


RESULTADOS DEL ANÁLISIS

1. Distancias calculadas a cada vecino (k=3):
   Vecino 1: distancia = 1.4194
   Vecino 2: distancia = 1.7584
   Vecino 3: distancia = 1.9854

2. Vecinos seleccionados (índices):
   [1, 4, 2]

3. Detalles de los vecinos más cercanos:

   Vecino 1 (índice 1, distancia 1.4194):
     - Sexo: 2
     - Ciudad: Quito
     - Colesterol: alto
     - Edad: 52
     - Diabetes: si (1)

   Vecino 2 (índice 4, distancia 1.7584):
     - Sexo: 2
     - Ciudad: Ambato
     - Colesterol: medio
     - Edad: 45
     - Diabetes: no (0)

   Vecino 3 (índice 2, distancia 1.9854):
     - Sexo: 2
     - Ciudad: Guayaquil
     - Colesterol: medio
     - Edad: 34
     - Diabetes: no (0)


### Predicción Final

Mostramos la predicción final basada en la votación mayoritaria de los 3 vecinos más cercanos.

In [14]:
# Votación mayoritaria
votes = [df.iloc[idx]['diabetes_bin'] for idx in neighbors_idx]
mean_vote = np.mean(votes)

print("\n" + "="*80)
print("VOTACIÓN MAYORITARIA")
print("="*80)

print(f"\nVotos de los 3 vecinos: {votes}")
print(f"Promedio de votos: {mean_vote:.3f}")
print(f"Redondeado: {round(mean_vote)}")

print(f"\n" + "="*80)
print(f"PREDICCIÓN FINAL")
print(f"="*80)
print(f"\nCódigo de predicción: {pred}")
if pred == 0:
    prediction_text = "NO TIENE DIABETES"
else:
    prediction_text = "TIENE DIABETES"

print(f"Predicción: El nuevo paciente {prediction_text}")
print(f"\n{'='*80}")


VOTACIÓN MAYORITARIA

Votos de los 3 vecinos: [np.int64(1), np.int64(0), np.int64(0)]
Promedio de votos: 0.333
Redondeado: 0

PREDICCIÓN FINAL

Código de predicción: 0
Predicción: El nuevo paciente NO TIENE DIABETES



## Resumen

En este notebook hemos:

1. **Instalado dependencias**: pandas, numpy y scikit-learn
2. **Implementado FeatureTransformer**: clase que transforma datos categóricos y numéricos
3. **Calculado distancia euclídea**: entre puntos en espacio multidimensional
4. **Implementado KNN**: algoritmo de K-vecinos más cercanos
5. **Cargado los datos**: dataset de pacientes ecuatorianos
6. **Preparado el target**: convertir diabetes de texto a binario
7. **Transformado características**: one-hot encoding y estandarización
8. **Ejecutado KNN**: encontrar 3 vecinos más cercanos
9. **Realizado votación**: determinar predicción por mayoría
10. **Presentado resultados**: distancias, vecinos y predicción final

### Conceptos Clave

- **One-hot encoding**: Convertir categorías en columnas binarias (0/1)
- **Estandarización**: Escalar datos a media 0 y desviación estándar 1
- **Distancia euclídea**: Medida de similitud entre puntos
- **KNN**: Algoritmo que clasifica basado en los k vecinos más cercanos
- **Votación mayoritaria**: Decisión basada en la mayoría de votos